# Air Quality Forecasting — Exploratory Data Analysis

This notebook explores the **Beijing PM2.5** hourly air-quality dataset (UCI, 2010–2014). The goal is to understand the structure of the data, the distribution and missingness of the target pollutant `pm2.5`, its seasonal/daily patterns, and how it relates to the meteorological variables (dew point, temperature, pressure, wind, snow, rain). These findings drive the feature engineering choices in notebook 02.

## 1. Imports & Load Data

We load the raw hourly dataset and build a single `Datetime` column from the `year`, `month`, `day`, and `hour` integer columns so that pandas can work with it as a proper time index.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
%matplotlib inline

df = pd.read_csv('data/PRSA_data.csv')
df['Datetime'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
df = df.sort_values('Datetime').reset_index(drop=True)
df.head()

## 2. Dataset Overview

A quick look at shape, data types, basic statistics, duplicates, and the date range gives us a sense of the data's coverage and quality. The dataset holds one row per hour for five years.

In [ ]:
print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)
print('\nDate range:', df['Datetime'].min(), '→', df['Datetime'].max())
print('Duplicate timestamps:', df['Datetime'].duplicated().sum())
print('\nWind direction (cbwd) categories:')
print(df['cbwd'].value_counts())

In [ ]:
df.describe().round(2)

## 3. Missing & Invalid Values Analysis

The target `pm2.5` contains a large number of missing readings (the sensor was offline for stretches of time). The meteorological columns are complete. Because we cannot impute the target without fabricating ground truth, rows with a missing `pm2.5` will be **dropped** during cleaning.

In [ ]:
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
miss_tbl = pd.DataFrame({'missing': missing, 'pct': missing_pct})
print(miss_tbl[miss_tbl['missing'] > 0])

plt.figure(figsize=(9, 4))
miss_tbl[miss_tbl['missing'] > 0]['pct'].plot(kind='bar', color='tomato', edgecolor='black')
plt.ylabel('% missing'); plt.title('Missing Values by Column')
plt.tight_layout(); plt.show()

## 4. Target Variable Distribution

A histogram and box plot of `pm2.5` show the overall spread, central tendency, and the heavy right skew typical of pollution data: most hours are moderate, but there is a long tail of severe smog episodes reaching several hundred µg/m³.

In [ ]:
pm = df['pm2.5'].dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(pm, bins=60, color='steelblue', edgecolor='black')
axes[0].set_xlabel('pm2.5 (µg/m³)'); axes[0].set_ylabel('Count')
axes[0].set_title('PM2.5 Distribution')
axes[1].boxplot(pm, vert=False)
axes[1].set_xlabel('pm2.5 (µg/m³)'); axes[1].set_title('PM2.5 Box Plot')
plt.tight_layout(); plt.show()

print(pm.describe().round(2))
print('Skew:', round(pm.skew(), 2))

## 5. PM2.5 Over Time — Full Series

Plotting the entire five-year series (daily mean, for readability) reveals the strong annual cycle: winter heating season produces the worst air quality, while summers are comparatively clean.

In [ ]:
daily = df.set_index('Datetime')['pm2.5'].resample('D').mean()
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(daily.index, daily.values, linewidth=0.6, color='steelblue')
ax.set_title('Daily Mean PM2.5 — Beijing 2010–2014')
ax.set_xlabel('Date'); ax.set_ylabel('pm2.5 (µg/m³)')
plt.tight_layout(); plt.show()

## 6. Seasonality Analysis

Air pollution is shaped by time-of-day, day-of-week, and month-of-year. We extract these calendar components and plot the average `pm2.5` for each — confirming the strong monthly (heating-season) cycle and a milder daily rhythm.

In [ ]:
df['hour']      = df['Datetime'].dt.hour
df['dayofweek'] = df['Datetime'].dt.dayofweek
df['month']     = df['Datetime'].dt.month

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
df.groupby('hour')['pm2.5'].mean().plot(ax=axes[0], marker='o', color='steelblue')
axes[0].set_title('Mean PM2.5 by Hour of Day'); axes[0].set_ylabel('pm2.5 (µg/m³)')
df.groupby('dayofweek')['pm2.5'].mean().plot(ax=axes[1], marker='o', color='seagreen')
axes[1].set_title('Mean PM2.5 by Day of Week (0=Mon)')
df.groupby('month')['pm2.5'].mean().plot(ax=axes[2], marker='o', color='tomato')
axes[2].set_title('Mean PM2.5 by Month')
plt.tight_layout(); plt.show()

## 7. Correlation Analysis

A correlation heatmap over the numeric weather variables and `pm2.5` shows which meteorological drivers move with pollution. Dew point (`DEWP`) tends to be positively associated (humid, stagnant air traps particulates) while cumulated wind speed (`Iws`) is negatively associated (wind disperses pollution).

In [ ]:
num_cols = ['pm2.5', 'DEWP', 'TEMP', 'PRES', 'Iws', 'Is', 'Ir']
corr = df[num_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('Correlation Heatmap')
corr['pm2.5'].drop('pm2.5').sort_values().plot(kind='barh', color='steelblue', ax=axes[1])
axes[1].set_title('Correlation with PM2.5'); axes[1].set_xlabel('Pearson r')
plt.tight_layout(); plt.show()

## 8. Wind Direction vs PM2.5

`cbwd` is a categorical wind-direction code (NW, NE, SE, cv = calm/variable). Wind direction matters because pollution is advected from different source regions. A bar chart of mean `pm2.5` per direction shows which winds bring the dirtiest air to Beijing.

In [ ]:
wind_means = df.groupby('cbwd')['pm2.5'].agg(['mean', 'count']).round(1)
print(wind_means)

plt.figure(figsize=(8, 4))
df.groupby('cbwd')['pm2.5'].mean().sort_values().plot(kind='bar', color='seagreen', edgecolor='black')
plt.ylabel('Mean pm2.5 (µg/m³)'); plt.title('Mean PM2.5 by Wind Direction')
plt.tight_layout(); plt.show()

## 9. Sample Week — Zoomed-In View

A single week of hourly data shows the short-term autocorrelation of pollution: PM2.5 builds up over hours/days of stagnant weather and then clears rapidly when a weather front arrives. This persistence is exactly what lag and rolling-mean features will exploit.

In [ ]:
sample_week = df[(df['Datetime'] >= '2014-01-13') & (df['Datetime'] < '2014-01-20')]
plt.figure(figsize=(14, 4))
plt.plot(sample_week['Datetime'], sample_week['pm2.5'], marker='.', color='steelblue')
plt.title('PM2.5 — One Sample Week (Jan 2014)')
plt.xlabel('Datetime'); plt.ylabel('pm2.5 (µg/m³)')
plt.tight_layout(); plt.show()

## 10. Summary of Key Findings

The table below captures the most important observations from the EDA. These guide the feature engineering choices in notebook 02 (calendar features for seasonality, lag/rolling features for persistence, one-hot wind direction).

In [ ]:
summary = pd.DataFrame({
    'Finding': [
        'Dataset size',
        'Date range',
        'Missing pm2.5 rows',
        'Target mean (µg/m³)',
        'Target median (µg/m³)',
        'Target max (µg/m³)',
        'Strongest weather corr (+)',
        'Strongest weather corr (−)',
        'Worst month',
        'Cleanest wind direction',
    ],
    'Value': [
        f'{len(df):,} hourly rows',
        f"{df['Datetime'].min().date()} → {df['Datetime'].max().date()}",
        f"{df['pm2.5'].isna().sum():,} ({df['pm2.5'].isna().mean()*100:.1f}%)",
        f"{df['pm2.5'].mean():.1f}",
        f"{df['pm2.5'].median():.1f}",
        f"{df['pm2.5'].max():.0f}",
        f"DEWP (r={df[['pm2.5','DEWP']].corr().iloc[0,1]:.2f})",
        f"Iws  (r={df[['pm2.5','Iws']].corr().iloc[0,1]:.2f})",
        f"month {df.groupby('month')['pm2.5'].mean().idxmax()}",
        f"{df.groupby('cbwd')['pm2.5'].mean().idxmin()}",
    ],
})
summary